In [ ]:
def gerar_dataset_e_grafico_heatmap_quem_sao_os_inscritos_contratados():

    from constantes import pasta_data_04_load_database, pasta_data_05_processed

    import pandas as pd
    import numpy as np
    from pathlib import Path
    import matplotlib.pyplot as plt
    import seaborn as sns
    import re

    # ==============================================================================
    # 1. CONFIGURAÇÕES
    # ==============================================================================

    pd.set_option('display.max_rows', 100)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.float_format', '{:.2f}'.format)

    sns.set_theme(style="white", font_scale=1.1)

    caminho_final_parquet = pasta_data_05_processed / 'analise_006_decomposicao_status.parquet'
    pasta_figuras = Path('../reports/figures/analise006_1_1').resolve()
    pasta_figuras.mkdir(parents=True, exist_ok=True)

    # ==============================================================================
    # 2. LOAD + FILTRO
    # ==============================================================================

    print("[*] Carregando base...")
    caminho_base = pasta_data_04_load_database / 'inscritos_final_limpo.parquet'
    df = pd.read_parquet(str(caminho_base))

    df['modalidade_fies'] = df['modalidade_fies'].astype(str).str.strip().str.upper()
    df = df[(df['opcao_curso'] == 1) & (df['modalidade_fies'] == 'MODALIDADE I')]

    print(f"Total após filtro: {len(df)}")

    # ==============================================================================
    # 3. FAIXAS (SEM PERDA DE DADOS)
    # ==============================================================================

    df['nome_cine_area_geral'] = df['nome_cine_area_geral'].fillna('CINE NÃO INFORMADO')

    bins_renda = [-np.inf, 600, 1200, 1800, 2400, 3000, np.inf]
    labels_renda = [
        '0-600',
        '601-1200',
        '1201-1800',
        '1801-2400',
        '2401-3000',
        '> 3000'
    ]
    df['faixa_renda_bruta'] = pd.cut(df['renda_per_capita'], bins=bins_renda, labels=labels_renda)

    df['gap'] = df['media_enem'] - df['nota_corte_gp']

    bins_gap = [-np.inf, -150, -50, 0, 50, 150, np.inf]
    labels_gap = [
        '< -150',
        '[-150, -50]',
        '[-50, 0]',
        '[0, +50]',
        '[+50, +150]',
        '> +150'
    ]
    df['nivel_nota_gap'] = pd.cut(df['gap'], bins=bins_gap, labels=labels_gap)

    # ==============================================================================
    # 4. STATUS (100% CONTROLADO)
    # ==============================================================================

    def normalizar(s):
        return str(s).strip().upper()

    mapa_status = {
        'CONTRATADA': '1. CONTRATADA',
        'INSCRIÇÃO POSTERGADA': '2. INSCRIÇÃO POSTERGADA',
        'PRE-SELECIONADO': '3. PRÉ-SELECIONADO',
        'PRÉ-SELECIONADO': '3. PRÉ-SELECIONADO',
        'NÃO CONTRATADO': '4. NÃO CONTRATADO',
        'NAO CONTRATADO': '4. NÃO CONTRATADO',
        'REJEITADA PELA CPSA': '5. REJEITADA PELA CPSA',
        'OPÇÃO NÃO CONTRATADA': '6. OPÇÃO NÃO CONTRATADA',
        'OPCAO NAO CONTRATADA': '6. OPÇÃO NÃO CONTRATADA',
        'PARTICIPACAO CANCELADA PELO CANDIDATO': '7. PARTICIPACAO CANCELADA',
        'PARTICIPAÇÃO CANCELADA PELO CANDIDATO': '7. PARTICIPACAO CANCELADA',
        'LISTA DE ESPERA': '8. LISTA DE ESPERA'
    }

    df['status_norm'] = df['situacao_fies'].apply(normalizar)
    df['status_final'] = df['status_norm'].map(mapa_status)

    nao_mapeados = df[df['status_final'].isna()]['status_norm'].value_counts()

    if not nao_mapeados.empty:
        print("⚠️ Status não mapeados:")
        print(nao_mapeados.head(20))

    df['status_final'] = df['status_final'].fillna('9. OUTROS')

    # ==============================================================================
    # 5. CHECK DE PERDA DE DADOS
    # ==============================================================================

    print("\n🔎 CHECK DE INTEGRIDADE:")
    print("Sem renda:", df['faixa_renda_bruta'].isna().sum())
    print("Sem nota:", df['nivel_nota_gap'].isna().sum())
    print("Sem status:", df['status_final'].isna().sum())

    # ==============================================================================
    # 6. MATRIZ (CUBO CORRETO)
    # ==============================================================================

    df_counts = df.groupby(
        ['nome_cine_area_geral', 'faixa_renda_bruta', 'nivel_nota_gap', 'status_final'],
        observed=True
    ).size().reset_index(name='qtd')

    df_totals = df.groupby(
        ['nome_cine_area_geral', 'faixa_renda_bruta', 'nivel_nota_gap'],
        observed=True
    ).size().reset_index(name='total_celula')

    df_analise = df_counts.merge(
        df_totals,
        on=['nome_cine_area_geral', 'faixa_renda_bruta', 'nivel_nota_gap']
    )

    df_analise['percentual_celula'] = (
        df_analise['qtd'] / df_analise['total_celula'] * 100
    )

    df_analise.to_parquet(str(caminho_final_parquet), index=False)

    # ==============================================================================
    # 7. AUDITORIA TOTAL (TODAS AS CÉLULAS)
    # ==============================================================================

    print("\n🔍 AUDITORIA COMPLETA:")

    check = df_analise.groupby(
        ['nome_cine_area_geral', 'faixa_renda_bruta', 'nivel_nota_gap'],
        observed=True
    )['percentual_celula'].sum().reset_index()

    erros = check[~np.isclose(check['percentual_celula'], 100.0, atol=0.01)]

    print(f"Total de células: {len(check)}")
    print(f"Células com erro: {len(erros)}")

    if not erros.empty:
        print("⚠️ ERROS ENCONTRADOS:")
        print(erros.head(20))
    else:
        print("✅ Todas as células somam 100%")

    # ==============================================================================
    # 8. PLOT (FORMATADO)
    # ==============================================================================

    ordem_nota_plot = labels_gap[::-1]

    def gerar_mapas(titulo, dados):

        status_paineis = [
            ('1. CONTRATADA', 'Contratados'),
            ('4. NÃO CONTRATADO', 'Não contratados')
        ]

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(26, 11.2),
            dpi=350,
            sharey=True,
            gridspec_kw={
                'wspace': 0.035,
                'width_ratios': [1, 1]
            }
        )

        for i, (status, titulo_status) in enumerate(status_paineis):

            ax = axes[i]

            df_status = dados[dados['status_final'] == status]

            if df_status.empty:
                matriz = pd.DataFrame(
                    0,
                    index=ordem_nota_plot,
                    columns=labels_renda
                )
            else:
                matriz = df_status.pivot(
                    index='nivel_nota_gap',
                    columns='faixa_renda_bruta',
                    values='percentual_celula'
                )

                matriz = matriz.reindex(
                    index=ordem_nota_plot,
                    columns=labels_renda
                ).fillna(0)

            sns.heatmap(
                matriz,
                annot=True,
                fmt=".1f",
                cmap="magma_r",
                vmin=0,
                vmax=100,
                linewidths=0.45,
                linecolor='white',
                annot_kws={
                    "size": 29,
                    "weight": "bold",
                    "color": "black"
                },
                cbar=False,
                square=False,
                ax=ax
            )

            # sem título
            ax.set_title("")

            ax.set_xlabel(
                'Faixa de Renda Familiar Per Capita',
                fontweight='bold',
                fontsize=22,
                color='black',
                labelpad=10
            )

            ax.set_xticklabels(
                ax.get_xticklabels(),
                rotation=0,
                fontsize=22,
                color='black'
            )

            ax.tick_params(axis='x', colors='black', pad=2)

            if i == 0:
                ax.set_ylabel(
                    'Desempenho (Gap da Nota)',
                    fontweight='bold',
                    fontsize=22,
                    color='black',
                    labelpad=10
                )

                ax.set_yticklabels(
                    ax.get_yticklabels(),
                    rotation=0,
                    fontsize=22,
                    color='black'
                )

                ax.tick_params(axis='y', colors='black', pad=2)

            else:
                ax.set_ylabel('')
                ax.tick_params(axis='y', left=False, labelleft=False)

        plt.subplots_adjust(
            left=0.07,
            right=0.995,
            bottom=0.11,
            top=0.995,
            wspace=0.035
        )

        n_clean = re.sub(r'\W+', '_', titulo.lower())

        plt.savefig(
            pasta_figuras / f"{n_clean}_contratados_nao_contratados.png",
            dpi=700,
            bbox_inches='tight',
            pad_inches=0.06
        )

        plt.close()

    # Nacional
    df_nac = df_analise.groupby(
        ['faixa_renda_bruta', 'nivel_nota_gap', 'status_final'],
        observed=True
    )['qtd'].sum().reset_index()

    df_nac_total = df_nac.groupby(
        ['faixa_renda_bruta', 'nivel_nota_gap'],
        observed=True
    )['qtd'].sum().reset_index(name='total')

    df_nac = df_nac.merge(
        df_nac_total,
        on=['faixa_renda_bruta', 'nivel_nota_gap']
    )

    df_nac['percentual_celula'] = df_nac['qtd'] / df_nac['total'] * 100

    gerar_mapas("NACIONAL", df_nac)

    # Por área
    for area in df_analise['nome_cine_area_geral'].unique():
        print(f"[*] Área: {area}")
        gerar_mapas(
            area,
            df_analise[df_analise['nome_cine_area_geral'] == area]
        )

    print("\n✅ FINALIZADO COM SUCESSO")


gerar_dataset_e_grafico_heatmap_quem_sao_os_inscritos_contratados()

[*] Carregando base...
Total após filtro: 1102122

🔎 CHECK DE INTEGRIDADE:
Sem renda: 0
Sem nota: 3
Sem status: 0

🔍 AUDITORIA COMPLETA:
Total de células: 350
Células com erro: 0
✅ Todas as células somam 100%
[*] Área: Agricultura, silvicultura, pesca e veterinária
[*] Área: Artes e humanidades
[*] Área: Ciências naturais, matemática e estatística
[*] Área: Ciências sociais, comunicação e informação
[*] Área: Computação e Tecnologias da Informação e Comunicação (TIC)
[*] Área: Educação
[*] Área: Engenharia, produção e construção
[*] Área: Negócios, administração e direito
[*] Área: Saúde e bem-estar
[*] Área: Serviços

✅ FINALIZADO COM SUCESSO
